In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import os
import random
import sys
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

In [2]:
SEED = 31
TRIALS = 200
TARGET = 'scalar_coupling_constant'
PREDICTION = 'pred'

In [3]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
seed_everything(SEED)

# Eval function

In [4]:
def group_mean_log_mae(y_true, y_pred, types, floor=1e-9):
    """
    Fast metric computation for this competition: https://www.kaggle.com/c/champs-scalar-coupling
    Code is from this kernel: https://www.kaggle.com/uberkinder/efficient-metric
    """
    maes = (y_true-y_pred).abs().groupby(types).mean()
    maes = np.log(maes.map(lambda x: max(x, floor)))
    return maes.mean()

In [5]:
lasso = pd.read_csv(f'../input/champs-scalar-coupling-lasso/submission.csv')
rf = pd.read_csv(f'../input/champs-scalar-coupling-rf/submission.csv')
xgb = pd.read_csv(f'../input/champs-scalar-coupling-xgb/submission.csv')
lgb = pd.read_csv(f'../input/champs-scalar-coupling-lgb/submission.csv')
test_sets = [lasso, rf, xgb, lgb]
print(f'lasso={lasso.shape}, rf={rf.shape}, xgb={xgb.shape}, lgb={lgb.shape}')

FileNotFoundError: [Errno 2] No such file or directory: '../input/champs-scalar-coupling-lasso/submission.csv'

In [6]:
lasso_train = pd.read_csv(f'../input/champs-scalar-coupling-lasso/train.csv')
rf_train = pd.read_csv(f'../input/champs-scalar-coupling-rf/train.csv')
xgb_train = pd.read_csv(f'../input/champs-scalar-coupling-xgb/train.csv')
lgb_train = pd.read_csv(f'../input/champs-scalar-coupling-lgb/train.csv')
train_sets = [lasso_train, rf_train, xgb_train, lgb_train]
print(f'Train sets\nlasso={lasso_train.shape}, rf={rf_train.shape}, xgb={xgb_train.shape}, lgb={lgb_train.shape}')

FileNotFoundError: [Errno 2] No such file or directory: '../input/champs-scalar-coupling-lasso/train.csv'

# Trials

In [7]:
%%time

def weights(n, min_weight=0.01, max_weight=0.99):
    if n < 1:
        raise ValueError('n must not be less than 1')
    res = []
    remainder = 1
    for _ in range(n - 1):
        w = random.uniform(min_weight, max_weight) * remainder
        res.append(w)
        remainder -= w
    res.append(remainder)
    return res


def trial(train_sets, prediction_column, target_column):
    ws = weights(len(train_sets))
    df = train_sets[0].copy()
    df[prediction_column] = 0
    for i, t in enumerate(train_sets):
        df[prediction_column] += t[prediction_column] * ws[i]
    score = group_mean_log_mae(df[target_column], df[prediction_column], df['type'])
    return score, ws


best = sys.maxsize
best_weights = []
for i in range(TRIALS):
    score, ws = trial(train_sets=train_sets, prediction_column=PREDICTION, target_column=TARGET)
    if score < best:
        best = score
        best_weights = ws
        
print(f'best={best:.4f}')
print(f'''best weights (sum={sum(best_weights)})
  lasso={best_weights[0]:.4f}
  rf={best_weights[1]:.4f}
  xgb={best_weights[2]:.4f}
  lgb={best_weights[3]:.4f}
''')

NameError: name 'train_sets' is not defined

In [8]:
submission = test_sets[0].copy()
submission[TARGET] = 0
for i, t in enumerate(test_sets):
    submission[TARGET] += t[TARGET] * best_weights[i]
submission.head()

NameError: name 'test_sets' is not defined

In [9]:
submission.to_csv('submission.csv', index=False)
print(os.listdir("."))

NameError: name 'submission' is not defined